# Reflections:

The three implementations we have included represent three distinct stages of algorithmic maturity:
    Implementation 1:  Introductory, pedagogical Smith–Waterman
    Implementation 2:  Biologically‑faithful, tie‑breaking‑aware Smith–Waterman
    Implementation 3 — High‑performance, Numba‑accelerated dynamic programming and traceback engine

Each stage solves a specific limitation of the previous one.
Providing all 3 implementations reflects the shift in goals we experienced as we worked our way through this project: from correctness, to biological realism, to scalability.

### Implementation 1:

This was our primary implementation, and was focused on pure python, with no optimization techniques so that we could gain an understanding of how to implement the Smith-Waterman logic and how the program would run. We started with a straightforward dynamic programming construction of the scoring matrix where for each cell, the algorithm computes the diagonal score, up score, and left score, then takes the max these values or zero. A simple helper function is used to decide which direction produced the score. The direction code (0,1,2,3) is stored in a matrix, but the matrix is not computed from score comparisons during traceback. Traceback simply follows the stored direction codes, stopping when the code is 0. Tie‑breaking is naïve: diagonal always wins ties, even when biologically incorrect. Traceback stops when the direction code is 0 (END).

This implementation works properly for simple examples and is easy to follow for instructional/learning purposes. It also correctly implements the local alignment zero‑reset rule. The program ran quickly, even with longer sequences, as long as there were no tied scores or complex alignments. There is clean separation between scoring and traceback direction assignment, and it is suitable for demonstrating the mechanics of matrix construction in dynamic programming.

However, we found that it would fail when testing with more complex sequences such as those with ambiguous diagonals, multiple competing local maxima, repetitive sequences, long local alignments with internal gaps, and long internal motifs surrounded by noise. Upon examination, we realized that without a true traceback matrix, the stored directions reflected the helper function's decision rather than the DP logic. This could result in traceback following a path that is not actually optimal according to the scoring matrix, producing biologically incorrect alignments.

We also realized that the diagonal was incorrectly favored in ties. With repetitive or ambiguous regions, the implementation was inventing diagonal matches that should not have existed, inflating alignment scores and producing false alignments. This would cause misalignment for biologically important motifs in repetitive regions, such as microsatellites. Repeats create many equal score paths, which the helper function cannot distinguish between correctly. This could cause alignments to become nondeterministic and biologically meaningless. Traceback was also only stopping when the direction was equal to zero, not when the score reached zero, and termination was dependent upon the helper function, not the matrix. This could cause traceback to stop too early or too late, truncating real alignments or extending into noise.

Because every cell is computed in python, one at a time, the runtime for this program grows quadratically. This causes even moderately long sequences to become impractical, making the implementation unsuitable for real biological data.

Ultimately, our initial implementation produces incorrect alignments in many realistic scenarios due to diagonal bias, incorrect termination, and non-matrix-based traceback. To obtain biologically meaningful results, we needed to address proper tie-breaking, score-based termination, and deterministic behavior, which is why we moved on to Implementation 2. We elected to keep this version inlcuded in the project, however, for its strengths in illustrating the fundamental concepts of Smith-Waterman.


### Implementation 2:

In this version, we addressed the conceptual flaws of Implementation 1. It computes scores correctly, uses strict tie‑breaking rules, and bases traceback termination on the score matrix rather than direction codes. DIAG is chosen only when strictly optimal, and UP versus LEFT ties are resolved by comparing neighboring scores, which yields deterministic behavior in ambiguous or repetitive regions.

We were able to agree that this was a biologically accurate implementation. The traceback stops at score = 0, matching the true local alignment rules. Tie breaking is not biased, but rather, follows a more deterministic and biologically rational resolution. We also aligned the sequences using lists, then reverse them at the end, which we found was an efficient way to track the alignment. We found that this implementation could handle ambiguous, repetitive, and noisy regions within the sequences correctly.

The dynamic programming recurrence, where we take the max of zero, the diagonal score + match/mismatch, left score + gap penalty, or up score + gap penalty is precisely how Smith-Waterman define it. Correct scoring is the foundation of all downstream decisions, and if the scoring logic is not implemented correctly, everything else fails. This assurance provided correct scoring of matches, mismatches, and gaps, with proper zero-reset boundaries and predictable local alignment behavior.

Even so, we were realizing during our testing that this version still had its own weaknesses. Because we set up the dynamic programming matrix to be filled cell‑by‑cell using Python loops, each iteration required the interpreter to perform dynamic type checks, memory lookups, and object creation. This overhead was negligible for short sequences but became prohibitive as sequence length increased.

Smith–Waterman has a time complexity of O(mn), so even modest increases in sequence length would cause the number of operations to grow quadratically. For example, aligning two sequences of length 10,000 would require 100 million cell updates, each executed through Python’s interpreted loop. As a result, Implementation 2 cannot scale to the sequence lengths commonly encountered in real biological datasets, such as long-read sequencing, large protein domains, or genomic windows.

A second limitation arises from the lack of vectorization or just‑in‑time compilation. Unlike languages such as C or compiled frameworks like Numba, Python does not take advantage of CPU‑level optimizations such as SIMD instructions, loop unrolling, or cache‑aware memory access. In this implementation, every cell update is executed as a high‑level Python operation, which prevents the CPU from optimizing the inner loop. Each iteration of that loop is executed by the Python interpreter, which is a high‑level virtual machine that processes instructions one at a time. Modern CPUs are designed to execute billions of low‑level operations per second, but Python cannot hand those operations directly to the CPU. Instead, Python must interpret each step, check types, allocate objects, and manage memory dynamically. This creates a massive overhead that dwarfs the actual arithmetic being performed. Even though each cell update is conceptually simple—just a few integer additions and comparisons—the interpreter adds hundreds of extra operations around it. As a result, the algorithm runs at a tiny fraction of the CPU’s actual capability.

This limitation prevents the algorithm from taking advantage of hardware features that compiled languages and JIT‑accelerated code can use automatically. Modern CPUs support SIMD (Single Instruction, Multiple Data) instructions, which allow the processor to perform the same arithmetic operation on multiple data points simultaneously. They also support pipelining, branch prediction, cache‑aware memory access, and loop unrolling. These optimizations can speed up dynamic programming by orders of magnitude, but Python cannot trigger them because the interpreter hides the underlying operations from the CPU. Therefore, every update becomes a slow, high‑level Python operation rather than a fast, low‑level machine instruction.

The inability to exploit hardware also affects memory access patterns. Dynamic programming is memory‑intensive: the algorithm repeatedly reads and writes to a large 2D matrix. Compiled code can optimize these accesses to fit into CPU caches, reducing the number of expensive RAM fetches. Python cannot do this. Each matrix access is a Python object lookup, not a raw memory access. This means the CPU spends most of its time waiting for Python to tell it what to do next, rather than performing the actual computation. As sequence lengths grow, this inefficiency compounds dramatically, making the algorithm unusable for large matrices.

Because of these limitations, Implementation 2 cannot scale to high‑throughput or large‑scale alignment tasks. High‑throughput tasks—such as aligning thousands of reads, scanning a genome for motifs, or comparing large protein families requires millions or billions of cell updates. In compiled code, this is feasible because the CPU can perform these operations at hardware speed, but in Python, each update is slowed by interpreter overhead, making the total runtime explode. Even if the algorithm is correct, it becomes practically unusable for real biological workloads. This is why Implementation 2 is limited to small or medium‑sized sequences and cannot be integrated into pipelines that require speed, scalability, or efficiency. The absence of compiled execution also prevents the algorithm from exploiting modern hardware capabilities, making it unsuitable for high‑throughput or large‑scale alignment tasks.

Traceback in Implementation 2 also suffers from Python’s performance limitations. Although the traceback logic is correct and deterministic, it relies on Python lists, Python branching, and Python recursion. These operations are significantly slower than their compiled equivalents. For long alignments—especially those containing extended gaps or long conserved regions—the traceback process becomes a bottleneck. Python’s recursion depth limits also impose practical constraints on the maximum alignment length that can be reconstructed safely. This makes the implementation fragile when dealing with long or complex alignments, even though the underlying logic is sound.

Memory usage presents another practical limitation. The scoring and traceback matrices must be stored in memory as full two‑dimensional Python‑managed NumPy arrays. While NumPy is efficient, the matrix still grows quadratically with sequence length. A 20,000 × 20,000 matrix contains 400 million cells, which requires several gigabytes of memory even when stored as 8‑byte integers. Python cannot efficiently allocate, manage, or operate on matrices of this size. This restricts Implementation 2 to relatively short sequences and prevents its use in genome‑scale analyses, long-read alignment, or large protein families.

Finally, Implementation 2 is not designed to explore multiple traceback paths or alternative alignments. It returns only a single optimal alignment, even in cases where the matrix contains multiple equally valid local maxima. This limits its usefulness in biological contexts where ambiguity is meaningful—for example, in repetitive regions, low‑complexity sequences, or protein families with multiple conserved motifs. The inability to enumerate alternative alignments restricts the algorithm’s interpretability and reduces its applicability to more nuanced biological questions.

Because we were curious about what a version would look like and run like that could scale to long sequences, high‑throughput workloads, and real‑world pipelines, we asked Eric to share his high-efficiency code with us, and we would like to share it with all of you.

### Implementation 3:

Implementation 3 is a high‑performance redesign of the Smith–Waterman/Needleman–Wunsch alignment algorithms that shifts from Python’s interpreted execution model to a compiled, hardware‑accelerated version. Instead of operating on Python strings and Python loops, this version encodes sequences numerically (A=1, C=2, G=3, T=4, N=5, -=6) and uses Numba’s just‑in‑time (JIT) compilation to convert the dynamic programming (DP) loop and traceback logic into optimized machine code. This allows the algorithm to run at speeds comparable to C or C++, while still being written in Python. Admittedly, this is much more advanced than the level at which we are currently coding, but we thought it would be informative to have Eric share his implementation with the class so we could see some optimization approaches for this project.

The DP matrix is constructed using a compiled function (fill_matrix_fast) that performs the core recurrence relation for either Smith–Waterman (local alignment) or Needleman–Wunsch (global alignment). Because the function is compiled, each DP cell update becomes a low‑level arithmetic operation rather than a Python bytecode instruction. The traceback phase is also redesigned: instead of Python recursion and lists, Implementation 3 uses Numba recursion and JitList, a dynamic array structure optimized for compiled execution. This enables the algorithm to explore multiple traceback paths efficiently and reconstruct alignments even when the matrix contains ambiguous or branching optimal paths.

Overall, Implementation 3 is engineered for speed, scalability, and throughput. It is capable of aligning long sequences, handling large DP matrices, and returning multiple valid alignments. This makes it suitable for real bioinformatics workloads such as motif discovery, long‑read alignment, and comparative genomics—tasks that are impossible to perform efficiently with pure Python implementations.

The most significant strength of Implementation 3 is its ability to execute the dynamic programming loop at machine‑level speed. By compiling the DP logic with Numba, the algorithm avoids Python’s interpreter overhead and allows the CPU to perform millions of cell updates per second. This transforms Smith–Waterman from a pedagogical demonstration into a practical tool capable of handling large biological datasets. The performance improvement is not incremental—it is transformative, often yielding speedups of 100× to 400× compared to pure Python.

Encoding sequences as integers rather than strings provides substantial performance benefits. Integer comparisons are constant‑time operations that Numba can optimize aggressively, while string comparisons require Python object lookups and dynamic type handling. Numeric encoding also reduces memory usage and improves cache locality, allowing the DP matrix to be processed more efficiently. This encoding step is essential for enabling Numba to generate optimized machine code.

The traceback logic in Implementation 3 uses Numba recursion and JitList, which are far more efficient than Python recursion and lists. This allows the algorithm to reconstruct long alignments without hitting Python’s recursion depth limits or suffering from slow list operations. It also enables the exploration of multiple traceback paths, which is important in biological contexts where ambiguous or repetitive regions may produce several equally valid alignments.

Implementation 3 is flexible enough to support both Smith–Waterman (local) and Needleman–Wunsch (global) alignment modes. This makes it more versatile than the other two implementations and allows it to be used in a wider range of bioinformatics applications. The mode is controlled by a simple parameter, and the dynamic programming logic adjusts accordingly.

Because of its speed and scalability, Implementation 3 can be used for tasks such as aligning long genomic regions, scanning sequences for motifs, or comparing large protein families. These tasks are impossible to perform efficiently with pure Python implementations. Implementation 3 bridges the gap between educational code and production‑level bioinformatics tools.

Despite this, we did uncover some weaknesses in this implementation. First, Numba‑compiled functions cannot be stepped through with a debugger and they cannot print intermediate states. Additionally, errors often appear at compile time rather than runtime, and the error messages can be cryptic. This makes the implementation significantly harder to debug, validate, or modify. For students or researchers who need to understand the algorithm’s internal behavior, this lack of transparency is a major drawback.

Second, Traceback arrays must be preallocated with a maximum length, so if the alignment contains more gaps than expected, the array may overflow or produce truncated alignments. Because Numba does not support dynamic resizing of NumPy arrays, the implementation must rely on JitList or over-sized buffers. This introduces the risk of silent errors if the buffer is too small or if the recursion depth exceeds expectations.

Third, Numba does not allow Python objects, dictionaries, or dynamic scoring tables inside compiled functions. This makes it difficult to implement more sophisticated scoring models such as affine gap penalties, substitution matrices, or context‑dependent scoring, which are all biologically important considerations when working with real data. The inability to incorporate them limits the algorithm’s flexibility, thereby restricting applicability.

Finally, Numba recursion is fast but difficult to reason. It is not always clear how many paths will be explored or how memory will be allocated during recursion. This can lead to unexpected behavior in ambiguous regions where multiple traceback paths exist. Indeed, we witnessed this exact problem with the implementation as the test sequences became more complex. Ensuring that all valid alignments are captured and that no invalid ones are produced requires careful testing and knowing how to adapt the code accordingly, which is challenging because it is not as explicit as regular python.

#### The key optimization techniques used in implementation 3 included:
    - JIT Compilation: Python loops converted into machine code, eliminating interpreter overhead.
    - Numeric Encoding: Slow string operations replaced with fast integer comparisons.
    - Memory‑Efficient DP Matrix: NumPy arrays with fixed types, allow Numba to optimize memory access.
    - Optimized Traceback: JitList and compiled recursion applied to avoid Python’s slow list operations and recursion limits.
    - Reduced Python Overhead: Minimizes Python involvement in the inner loops, allowing the CPU to apply hardware‑level optimizations.

These optimizations transform the algorithm from a demonstration into a high‑performance computational tool. A shift in priorities from clarity in Implementation 1 and biologically realistic decision‑making in Implementation 2 give way to speed and scalability in Implementation 3. This trade‑off is appropriate for real bioinformatics workloads where performance is essential and the algorithm must handle large datasets efficiently; however, the lack of explicit intermediate states, transparent control flow, and easily modifiable scoring logic makes the implementation far more difficult to debug, validate, or extend. In practice, this means that while Implementation 3 is the fastest and most scalable version we produced, it is also the least accessible for learning, experimentation, or fine‑grained biological modeling—highlighting the tension between performance and interpretability that often arises in computational biology.
